# Smoothing of an axial fiber-angle field

Tries `bayes_cep.statistics.axial_smoothing`, which smooths a per-vertex fiber-angle field on the surface mesh, with `smooth_axial_angle_field` (linear), `smooth_axial_angle_field_adaptive` or `smooth_axial_angle_field_median` (selected by `METHOD` below). A fiber direction is axial (a fiber and its negative are the same), so the field is smoothed as the doubled-angle unit vector $e^{2i\theta}$ by the heat equation on the surface, for a Gaussian of standard deviation `scale` (in mesh units); half the angle of the result is the smoothed angle. The adaptive variant multiplies the diffusivity by $1 + (|\nabla z|/g_0)^2$, so that narrow strips with a steep rotation of the fiber are removed quickly while smooth regions change little; $g_0$ is a quantile (`GRADIENT_QUANTILE`) of the gradients of the field that is smoothed. The median variant is a vector median filter on the doubled-angle vectors: every vertex takes the member of its neighborhood (`MEDIAN_RINGS` edges) closest to all others, `MEDIAN_ITERATIONS` times, followed by a linear smoothing with the `scale`. It removes structures narrower than about half the neighborhood (narrow strips, small rings) and keeps broad transitions; on a MAP estimate it left the fewest steep edges per change of the smooth regions of the three methods. The second output is the coherence $|z|\in[0,1]$: one where the neighboring directions agree, near zero where they cancel and the smoothed angle is not meaningful.

The notebook shows what the smoothing does to the field, how large the difference to the original is, and how a MAP estimate compares to the original and to the smoothed ground truth. This notebook is for interactive/visual inspection only.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv

from bayes_cep.mesh.interpolation import NearestNeighborInterpolationStrategy
from bayes_cep.mesh.io import load_pyvista_mesh
from bayes_cep.mesh.topology import get_triangle_connectivity
from bayes_cep.preprocessing.angle_field import build_angle_field_from_fiber_field
from bayes_cep.statistics.axial_error_metrics import (
    ERROR_THRESHOLDS_DEGREES,
    compute_axial_error_metrics,
)
from bayes_cep.statistics.axial_smoothing import (
    estimate_gradient_scale,
    smooth_axial_angle_field,
    smooth_axial_angle_field_adaptive,
    smooth_axial_angle_field_median,
)
from bayes_cep.statistics.axial_statistics import compute_axial_data_diff, wrap_axial_angles

pv.set_jupyter_backend("trame")

DATA_DIR = Path.cwd().parent / "example_data"
RAW_DIR = DATA_DIR / "raw"
PREPROCESSING_DIR = DATA_DIR / "preprocessing"
MAP_DIR = DATA_DIR / "map"

## Load the data

The real field is the per-vertex fiber angle of the example patient (`build_angle_field_from_fiber_field`); the synthetic one is the ground truth of the example preprocessing run, a sample of the prior. The synthetic field is smooth to begin with, so it shows what smoothing does to a field that has nothing to lose, the real one how much small-scale variation a real field has.

In [3]:
mesh = load_pyvista_mesh(RAW_DIR / "mesh.vtu")
fiber_field = np.load(RAW_DIR / "fiber_field.npy")
basis_vectors = np.load(RAW_DIR / "basis_vecs.npy")

real_angle_field = build_angle_field_from_fiber_field(mesh, fiber_field, basis_vectors)
synthetic_angle_field = np.load(PREPROCESSING_DIR / "ground_truth_angle_field.npy")

connectivity = get_triangle_connectivity(mesh)
vertex_to_simplex_matrix = NearestNeighborInterpolationStrategy().assemble_matrix(
    mesh.points, connectivity
)
surface_area = float(mesh.compute_cell_sizes().cell_data["Area"].sum())
print(f"{mesh.n_points} vertices, {mesh.n_cells} triangles, surface area {surface_area:.0f}")
print(f"spacing of 1000 randomly placed observations: about {np.sqrt(surface_area / 1000):.1f}")

15820 vertices, 31375 triangles, surface area 5205
spacing of 1000 randomly placed observations: about 2.3


## Settings

`FIELD` selects the field to smooth. `METHOD` selects the smoothing: `"linear"`, `"adaptive"` or `"median"`. `SCALES` are the standard deviations of the Gaussian smoothing in mesh units (for `"median"`, of the linear smoothing after the median filter; a scale of 0 leaves the field unchanged), for orientation: the spacing of 1000 observations is printed above, and the correlation length of the example prior is much larger (about 25 for `kappa=0.1`, `tau=5`).

In [4]:
FIELD = "real"  # "real" or "synthetic"
SCALES = (0.6, 1.5, 3.0, 6.0)
METHOD = "median"  # "linear", "adaptive" or "median"
GRADIENT_QUANTILE = 0.8  # adaptive: gradients above this quantile of the smoothed field are steep
MEDIAN_RINGS = 3  # median: neighborhood of the filter, in edges
MEDIAN_ITERATIONS = 2  # median: number of passes


def smooth(angles, scale):
    """Smooth a vertex angle field with the selected method.

    The gradient scale of the adaptive smoothing is a quantile of the gradients of that field
    itself. A scale of 0 returns the field wrapped onto (-pi/2, pi/2] for every method.
    """
    if METHOD == "linear" or scale == 0.0:
        return smooth_axial_angle_field(mesh, angles, scale)
    if METHOD == "adaptive":
        gradient_scale = estimate_gradient_scale(mesh, angles, GRADIENT_QUANTILE)
        return smooth_axial_angle_field_adaptive(mesh, angles, scale, gradient_scale)
    if METHOD == "median":
        return smooth_axial_angle_field_median(
            mesh, angles, MEDIAN_RINGS, MEDIAN_ITERATIONS, scale
        )
    raise ValueError(f"Unknown smoothing method {METHOD!r}.")


angle_field = {"real": real_angle_field, "synthetic": synthetic_angle_field}[FIELD]

## Smooth at several scales

For each scale: the mean and the fraction of vertices whose smoothed angle is within 10 degrees of the original, and the coherence. Vertices with a coherence near zero have no meaningful smoothed direction.

In [5]:
results = {scale: smooth(angle_field, scale) for scale in SCALES}

print(
    f"{'scale':>6}{'mean change [deg]':>20}{'within 10 deg':>16}"
    f"{'mean coherence':>16}{'coherence < 0.3':>17}"
)
for scale, (smoothed_angles, coherence) in results.items():
    change_degrees = np.degrees(np.abs(compute_axial_data_diff(smoothed_angles, angle_field)))
    print(
        f"{scale:>6}{change_degrees.mean():>20.1f}{np.mean(change_degrees < 10):>16.2f}"
        f"{coherence.mean():>16.2f}{np.mean(coherence < 0.3):>17.2f}"
    )

 scale   mean change [deg]   within 10 deg  mean coherence  coherence < 0.3
   0.6                15.9            0.53            0.91             0.01
   1.5                18.5            0.46            0.78             0.04
   3.0                22.1            0.36            0.59             0.12
   6.0                27.9            0.25            0.38             0.35


## The smoothed fields

Angle fields on simplices (the resolution the forward model sees), wrapped onto $(-\pi/2, \pi/2]$ with a cyclic color map. The original comes first; the coherence maps follow.

In [6]:
def show_simplex_field(vertex_values, title, cmap, clim, name):
    plot_mesh = mesh.copy()
    plot_mesh.cell_data[name] = vertex_to_simplex_matrix @ vertex_values
    plotter = pv.Plotter()
    plotter.add_mesh(plot_mesh, scalars=name, cmap=cmap, clim=clim)
    plotter.add_text(title)
    plotter.show()


angle_clim = (-np.pi / 2, np.pi / 2)
show_simplex_field(wrap_axial_angles(angle_field), f"{FIELD}: original", "hsv", angle_clim, "angle")
for scale, (smoothed_angles, _) in results.items():
    show_simplex_field(
        smoothed_angles, f"{FIELD}: smoothed, scale {scale}", "hsv", angle_clim, "angle"
    )

Widget(value='<iframe src="http://localhost:34367/index.html?ui=P_0x7f8b0e221d30_0&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:34367/index.html?ui=P_0x7f8afc7b8910_1&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:34367/index.html?ui=P_0x7f8afc7bbb10_2&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:34367/index.html?ui=P_0x7f8af53a2850_3&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:34367/index.html?ui=P_0x7f8af4724550_4&reconnect=auto" class="pyvi…

In [ ]:
for scale, (_, coherence) in results.items():
    show_simplex_field(coherence, f"coherence, scale {scale}", "viridis", (0.0, 1.0), "coherence")

## Difference between the smoothed and the original field

The cumulative distribution of the absolute axial difference per vertex: the fraction of vertices whose smoothed angle is closer than a given value to the original. The dotted lines mark the thresholds of the error metrics. This is the error a reconstruction at that resolution cannot get below when it is compared to the original field.

In [ ]:
fig, axis = plt.subplots(figsize=(6, 4))
for scale, (smoothed_angles, _) in results.items():
    sorted_change = np.sort(
        np.degrees(np.abs(compute_axial_data_diff(smoothed_angles, angle_field)))
    )
    axis.plot(
        sorted_change,
        np.arange(1, sorted_change.size + 1) / sorted_change.size,
        label=f"scale {scale}",
    )
for threshold in ERROR_THRESHOLDS_DEGREES:
    axis.axvline(threshold, color="gray", linestyle=":")
axis.set_xlabel("absolute axial difference to the original [deg]")
axis.set_ylabel("fraction of vertices below")
axis.set_ylim(0.0, 1.0)
axis.legend()
fig.tight_layout()

## A MAP estimate against the original and the smoothed ground truth

Compares a MAP estimate with the ground truth of the example preprocessing run, on simplices: against the original field, and against the ground truth smoothed at each scale (`TRUTH_SCALES`). A reconstruction that can only resolve a certain scale looks better against the smoothed truth. The last column also smooths the estimate itself at the same scale before comparing, which removes the small-scale noise of the estimate.

The MAP estimate is read from `example_data/map/` and the truth from `example_data/preprocessing/`; rerun `pixi run example` if the two do not belong together.

In [ ]:
TRUTH_SCALES = (0.0, 0.6, 1.5, 3.0)

map_estimate = np.load(MAP_DIR / "map_estimate.npy")
ground_truth = synthetic_angle_field

print(
    f"{'scale':>6} | {'estimate vs smoothed truth':^34} | "
    f"{'smoothed estimate vs smoothed truth':^36}"
)
print(
    f"{'':>6} | {'mean':>8}{'median':>9}{'p90':>8}{'<10 deg':>9} | "
    f"{'mean':>8}{'median':>9}{'p90':>8}{'<10 deg':>9}"
)
for scale in TRUTH_SCALES:
    smoothed_truth, _ = smooth(ground_truth, scale)
    smoothed_estimate, _ = smooth(map_estimate, scale)
    cells = []
    for estimate in (map_estimate, smoothed_estimate):
        metrics = compute_axial_error_metrics(
            compute_axial_data_diff(
                vertex_to_simplex_matrix @ estimate, vertex_to_simplex_matrix @ smoothed_truth
            )
        )
        cells.append(
            f"{metrics['error_mean_deg']:>8.1f}{metrics['error_median_deg']:>9.1f}"
            f"{metrics['error_p90_deg']:>8.1f}{metrics['fraction_below_10_deg']:>9.2f}"
        )
    print(f"{scale:>6} | {cells[0]} | {cells[1]}")

## Sign invariance

A fiber and its negative are the same fiber, so shifting any vertex angle by $\pi$ must not change the result.

In [ ]:
rng = np.random.default_rng(0)
shifted_angle_field = angle_field + np.pi * rng.integers(0, 2, size=angle_field.shape)
for scale in SCALES:
    smoothed_angles, _ = smooth(angle_field, scale)
    smoothed_shifted_angles, _ = smooth(shifted_angle_field, scale)
    difference_degrees = np.degrees(
        np.abs(compute_axial_data_diff(smoothed_angles, smoothed_shifted_angles))
    )
    print(f"scale {scale}: largest difference {difference_degrees.max():.1e} deg")